<a href="https://colab.research.google.com/github/tarabelo/PIAC-2627/blob/main/05%20-%20Efecto%20del%20ruido%20y%20barren%20plateaus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalamos qiskit en el notebook
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# **Efecto del ruido y *barren plateaus***

### Contenidos

1. [Ruido y canales cuánticos](#canales)
1. [Tipos de ruido](#tipos)
1. [Modelos de ruido en Qiskit Aer](#aer)
1. [Backends falsos (*fake backends*)](#fake)
1. [Ejecución en un sistema real de IBM](#ibm)
1. [QAOA con ruido](#qaoa)
    - [Ejercicio 8](#ejercicio8)
1. [*Barren plateaus*](#barren)

<a name="canales"></a>
## **Ruido y canales cuánticos**

Los ordenadores cuánticos disponibles actualmente pertenecen a la categoría de ordenadores *Noisy Intermediate-Scale Quantum*(NISQ) y destacan por imperfecciones en las puertas, la medida y en el aislamiento del sistema cuántico del exterior [arXiv:1801.00862](https://arxiv.org/abs/1801.00862).

En consecuencia, los estados generados en estos dispositivos son ruidoso y $|\psi\rangle$ pasa a representarse como una mezcla estadística de estados puros. Para representarlo usamos la **matriz densidad**:

$$\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i|, \qquad \rho = \rho^\dagger,\quad \text{Tr}(\rho)=1,\quad \rho \ge 0$$

- Un estado puro verifica $\rho = |\psi\rangle\langle\psi|$ y su *pureza* es $\text{Tr}(\rho^2) = 1$. Para un estado mezcla $\text{Tr}(\rho^2) < 1$
- El valor esperado de un observable es $\langle H\rangle = \text{Tr}(\rho H)$
- Una puerta unitaria transforma el estado como $\rho \rightarrow U\rho U^\dagger$

#### Canales cuánticos

La evolución más general de un sistema cuántico se describe mediante lo que se conoce como un **canal cuántico**: un mapa lineal $\mathcal{E}$ completamente positivo y que preserva la traza (CPTP) [Nielsen & Chuang](https://doi.org/10.1017/CBO9780511976667). Existen varias formas de representarlos, en este notebook usaremos la **representación de Kraus**:

$$
\mathcal{E}(\rho) = \sum_k K_k \rho K_k^\dagger, \qquad \sum_k K_k^\dagger K_k = I
$$

Los $K_k$ son los *operadores de Kraus*. Podemos interpretar el canal como si, con probabilidad $p_k = \text{Tr}(K_k\rho K_k^\dagger)$, se aplicara la operación $K_k$ sobre el estado.

Una puerta ideal es un caso particular de canal con un único operador de Kraus $K_0 = U$.

<a name="tipos"></a>
## **Tipos de ruido**

Existen muchos tipos de ruído en sistemas cuánticos, y la caracterización del ruído cuántico en si es un campo de investigación de alto impacto actualmente.

#### Canales de Pauli: *bit-flip* y *phase-flip*

El *bit-flip* aplica una puerta $X$ con una probabilidad $p$:

$$K_0 = \sqrt{1-p}\,I,\quad K_1 = \sqrt{p}\,X \quad\implies\quad \mathcal{E}(\rho) = (1-p)\rho + pX\rho X$$

El *phase-flip* aplica una puerta $Z$ con probabilidad $p$, intercambiando $|+\rangle \leftrightarrow |-\rangle$:

$$\mathcal{E}(\rho) = (1-p)\rho + pZ\rho Z$$

Se define un canal de Pauli, cualquier operación que puede ser descrita como: $\mathcal{E}(\rho) = p_I\rho + p_X X\rho X + p_Y Y\rho Y + p_Z Z\rho Z$.

#### Canal despolarizante

Con probabilidad $\lambda$ el estado de $n$ cúbits se sustituye por el estado totalmente mezclado:

$$\mathcal{E}(\rho) = (1-\lambda)\rho + \lambda\frac{I}{2^n}$$

Es el modelo más usado para describir de forma genérica el error medio de una puerta. Para un cúbit equivale a un canal de Pauli con $p_X=p_Y=p_Z=\lambda/4$.

#### Amortiguamiento de amplitud (relajación, $T_1$)

Modela la pérdida de energía del cúbit debido a la interacción con el entorno $|1\rangle \rightarrow |0\rangle$.

$$K_0 = \begin{pmatrix}1 & 0\\ 0 & \sqrt{1-\gamma}\end{pmatrix},\quad K_1 = \begin{pmatrix}0 & \sqrt{\gamma}\\ 0 & 0\end{pmatrix}, \qquad \gamma = 1-e^{-t/T_1}$$

#### Amortiguamiento de fase (desfase, $T_2$)

Modela la pérdida de coherencia del sistema:

$$K_0 = \begin{pmatrix}1 & 0\\ 0 & \sqrt{1-\lambda}\end{pmatrix},\quad K_1 = \begin{pmatrix}0 & 0\\ 0 & \sqrt{\lambda}\end{pmatrix}$$

Combinando ambos procesos (relajación térmica), un cúbit en reposo durante un tiempo $t$ evoluciona como:

$$
\rho = \begin{pmatrix}\rho_{00} & \rho_{01}\\ \rho_{10} & \rho_{11}\end{pmatrix}
\;\longrightarrow\;
\begin{pmatrix}1-\rho_{11}e^{-t/T_1} & \rho_{01}e^{-t/T_2}\\ \rho_{10}e^{-t/T_2} & \rho_{11}e^{-t/T_1}\end{pmatrix}, \qquad T_2 \le 2T_1
$$

Los tiempos $T_1$ y $T_2$ son los parámetros que se publican en la calibración de cualquier procesador cuántico, como es el caso de los [ordenadores de IBM](https://www.ibm.com/quantum/hardware).

#### Errores de lectura

Son errores derivados de la lectura de los cúbits.

#### Otros errores

- **Errores coherentes**: errores unitarios sistemáticos, como sobre-rotaciones $R_x(\theta + \epsilon)$. Al no ser aleatorios se acumulan de forma coherente
- **Crosstalk**: una operación sobre un cúbit afecta a sus vecinos.

#### Resumen y funciones de Qiskit Aer

| Ruido | Efecto | Función en `qiskit_aer.noise` |
|---|---|---|
| Bit-flip / phase-flip | Aplica $X$ / $Z$ con probabilidad $p$ | `pauli_error` |
| Despolarizante | Mezcla con el estado $I/2^n$ | `depolarizing_error` |
| Amortiguamiento de amplitud | Relajación $\vert 1\rangle\rightarrow\vert 0\rangle$ | `amplitude_damping_error` |
| Amortiguamiento de fase | Pérdida de coherencia | `phase_damping_error` |
| Relajación térmica | $T_1$ y $T_2$ durante el tiempo de una puerta | `thermal_relaxation_error` |
| Coherente | Unitario erróneo | `coherent_unitary_error` |
| Lectura | Matriz de confusión | `ReadoutError` |

A continuación visualizamos el efecto de estos ruidos sobre la esfera de bloch de un cúbit que esta inicialmente en el estado $|0\rangle$.

In [ ]:
from qiskit.quantum_info import DensityMatrix, Kraus

# Probabilidad de ruido
p = 0.2

# Canal bit-flip definido por sus operadores de Kraus y probabilidad p
# K0 = \sqrt{1-p} I, K1 = \sqrt{p} X
K0 = np.sqrt(1 - p) * np.eye(2)
K1 = np.sqrt(p) * np.array([[0, 1], [1, 0]])
bit_flip = Kraus([K0, K1])

# Matrix densidad estado puto |0>
rho = DensityMatrix.from_label('0')

# Evolucionamos la matriz por el canal
rho_ruido = rho.evolve(bit_flip)

display(rho.draw('latex',prefix='\\rho = '))
display(rho_ruido.draw('latex', prefix='\\rho_{noise} = '))
print("Pureza inicial Tr(ρ²) = {:.3f}, Pureza final Tr(ρ²) = {:.3f}".format(rho.purity().real, rho_ruido.purity().real))

In [ ]:
from qiskit_aer.noise import pauli_error, depolarizing_error

# Qiskit Aer permite crear estos canales

# Error bit-flip
error_bf = pauli_error([('X', p), ('I', 1 - p)])
print('Error bit-flip')
print(error_bf, '\n')

# Error phase-flip
error_pf = pauli_error([('Z', p), ('I', 1 - p)])
print('Error phase-flip')
print(error_pf, '\n')

# Canal despolarizante
error_dp = depolarizing_error(p, 1)
print('Error depolarizante')
print(error_dp)

In [ ]:
from qiskit.quantum_info import Pauli

# Efecto de cada canal sobre la esfera de Bloch: aplicamos el canal a todos
# los estados puros (superficie de la esfera) y representamos sus imágenes

# Convertimos los errores de Qiskit Aer (definidos en la celda anterior) en
# canales de quantum_info, que se pueden aplicar a una matriz densidad con .evolve()
canales = {
  'Bit-flip': error_bf.to_quantumchannel(),
  'Phase-flip': error_pf.to_quantumchannel(),
  'Despolarizante': error_dp.to_quantumchannel(),
}

# Matrices de Pauli X, Y, Z (las usamos como base para el vector de Bloch)
paulis = [Pauli('X'), Pauli('Y'), Pauli('Z')]

def estado_desde_bloch(r):
  # Construye la matriz densidad a partir de un vector de Bloch r = (rx, ry, rz)
  # ρ = (I + r·σ) / 2, con r·σ = rx·X + ry·Y + rz·Z
  return DensityMatrix((np.eye(2) + sum(ri * P.to_matrix() for ri, P in zip(r, paulis))) / 2)

def vector_bloch(rho):
  # Operación inversa: recupera el vector de Bloch de una matriz densidad
  # r = (<X>, <Y>, <Z>), donde <P> = Tr(ρ P); .real elimina la parte imaginaria numérica
  return [rho.expectation_value(P).real for P in paulis]

# Malla de vectores de Bloch sobre la esfera unidad
# phi: ángulo azimutal (30 valores, 0 a 2π); theta: ángulo polar (15 valores, 0 a π)
# meshgrid devuelve dos arrays de forma (15, 30)
phi, theta = np.meshgrid(np.linspace(0, 2 * np.pi, 30), np.linspace(0, np.pi, 15))

# Coordenadas cartesianas (x, y, z) de cada punto; stack con axis=-1 junta las tres
# coordenadas en un último eje, dando un array de forma (15, 30, 3)
esfera = np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], axis=-1)

# A la izquierda la esfera inicial (sin ruido) y a continuación su imagen tras cada canal
superficies = {'Estado inicial': esfera}
for nombre, canal in canales.items():
  # Para cada punto r de la malla (último eje): construimos la matriz de densidad ρ,
  # le aplicamos el canal con .evolve() y extraemos el nuevo vector de Bloch.
  # El resultado conserva la forma (15, 30, 3)
  superficies[nombre] = np.apply_along_axis(lambda r: vector_bloch(estado_desde_bloch(r).evolve(canal)), -1, esfera)

# Una figura ancha con 4 paneles 3D: estado inicial y los tres canales
fig = plt.figure(figsize=(20, 5))
for i, (nombre, superficie) in enumerate(superficies.items()):
  ax = fig.add_subplot(1, 4, i + 1, projection='3d')
  # Esfera original como malla gris tenue, de referencia
  # transpose(2, 0, 1) pasa de (15, 30, 3) a (3, 15, 30) y el * reparte x, y, z
  ax.plot_wireframe(*esfera.transpose(2, 0, 1), color='gray', alpha=0.2, linewidth=0.5)
  # Imagen de la esfera tras el canal, como superficie azul semitransparente
  ax.plot_surface(*superficie.transpose(2, 0, 1), color='#648fff', alpha=0.6)
  # Mismos límites y proporción 1:1:1 en todos los paneles para poder compararlos
  ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
  ax.set_box_aspect((1, 1, 1))
  ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
  # El primer panel no lleva p porque no tiene ruido
  ax.set_title(nombre if i == 0 else "{} (p = {})".format(nombre, p))
plt.show()

---



---



---

<a name="aer"></a>
# **Modelos de ruido en Qiskit Aer**

El módulo `qiskit_aer.noise` permite construir modelos de ruido a partir de tres elementos [Qiskit Aer: *Building Noise Models*](https://qiskit.github.io/qiskit-aer/tutorials/3_building_noise_models.html):

- `QuantumError`: canal cuántico asociado a una puerta. Se crea con las funciones anteriores y se pueden combinar con `.compose()` (secuencialmente) o `.tensor()` (sobre cúbits distintos)
- `ReadoutError`: error de lectura, a partir de la matriz $\begin{pmatrix}P(0|0) & P(1|0)\\ P(0|1) & P(1|1)\end{pmatrix}$
- `NoiseModel`: colección de errores asociados a instrucciones y cúbits:
  - `add_all_cúbit_quantum_error(error, instrucciones)`: el error se aplica tras esas puertas en todos los cúbits
  - `add_quantum_error(error, instrucciones, cúbits)`: solo en los cúbits indicados
  - `add_all_cúbit_readout_error(error)`: error de lectura en todos los cúbits

El modelo se pasa al simulador (`AerSimulator(noise_model=...)`) o a las primitivas de Aer.

## Ejemplo 1: Estado de Bell con ruido

Preparamos el estado $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ y lo simulamos sin ruido:

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.visualization import plot_histogram

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()
display(bell.draw('mpl'))

shots = 10000

# Usamos un Sampler sin ruido
sampler_ideal = AerSampler()
counts_ideal = sampler_ideal.run([bell], shots=shots).result()[0].data.meas.get_counts()
print(counts_ideal)

Definimos un modelo de ruido con error despolarizante en las puertas y error de lectura:

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

p1 = 0.01     # Error de las puertas de 1 cúbit
p2 = 0.05     # Error de las puertas de 2 cúbits
p_med = 0.03  # Probabilidad de leer el bit cambiado

modelo_ruido = NoiseModel()
modelo_ruido.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ['h', 'sx', 'x'])
modelo_ruido.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ['cx'])
modelo_ruido.add_all_qubit_readout_error(ReadoutError([[1 - p_med, p_med], [p_med, 1 - p_med]]))

print(modelo_ruido)

Simulamos el circuito con el modelo de ruido y comparamos ambas distribuciones :

In [ ]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator

sampler_ruido = AerSampler(options={"backend_options": {"noise_model": modelo_ruido}})

# Transpilamos al simulador con ruido: las puertas del circuito tienen que ser
# las que llevan error asociado en el modelo
pm = generate_preset_pass_manager(optimization_level=1,
                                  backend=AerSimulator(noise_model=modelo_ruido))
bell_ruido = pm.run(bell)

counts_ruido = sampler_ruido.run([bell_ruido], shots=shots).result()[0].data.meas.get_counts()
print(counts_ruido)
display(plot_histogram([counts_ideal, counts_ruido], legend=['Ideal', 'Con ruido']))

---



---



---

<a name="fake"></a>
# **Backends falsos (*fake backends*)**

El módulo `qiskit_ibm_runtime.fake_provider` incluye *backends falsos* que contienen una copia de la calibración de procesadores reales de IBM. Resultan realmente interesantes al incluir el mapa de conectividad, puertas base, $T_1$, $T_2$, errores de puerta, duraciones y errores de lectura; resultando en simulaciones con ruido realmente próximas a la realidad si la calibración es correccta.

- `NoiseModel.from_backend(backend)`: extrae el modelo de ruido del backend
- `AerSimulator.from_backend(backend)`: crea un simulador que reproduce el ruido (y las restricciones) del backend

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
import rustworkx as rx
from rustworkx.visualization import mpl_draw

backend_fake = FakeVigoV2()
print("cúbits:", backend_fake.num_qubits)
print("Puertas base:", backend_fake.operation_names)

# Parámetros de calibración del cúbit 0 (en segundos)
props = backend_fake.qubit_properties(0)
print("cúbit 0: T1 = {:.1f} µs, T2 = {:.1f} µs".format(props.t1 * 1e6, props.t2 * 1e6))

# Mapa de conectividad dibujado con rustworkx + matplotlib (sin necesidad de Graphviz)
grafo = backend_fake.coupling_map.graph.to_undirected(multigraph=False)
plt.figure(figsize=(8, 3))
mpl_draw(grafo, pos=rx.spring_layout(grafo, seed=1), with_labels=False,
         node_color='#648fff', font_color='white', node_size=800, width=3)
plt.show()

In [ ]:
# Mostramos los datos del modelo de ruido
modelo_fake = NoiseModel.from_backend(backend_fake)
print(modelo_fake)

In [ ]:
# Transpilamos al simulador con ruido: las puertas del circuito tienen que ser
# las que llevan error asociado en el modelo
pm_fake = generate_preset_pass_manager(optimization_level=1,
                                  backend=backend_fake)
# Circuito transpilado para el fake backend
bell_fake = pm_fake.run(bell)

# El Sampler de Aer para el fake backend
sampler_fake = AerSampler.from_backend(backend_fake)

counts_fake = sampler_fake.run([bell_fake], shots=shots).result()[0].data.meas.get_counts()
print(counts_fake)
display(plot_histogram([counts_ideal, counts_ruido, counts_fake],
                       legend=['Ideal', 'Modelo propio', 'Fake Backend']))

---



---



---

<a name="ibm"></a>
# **Ejecución en un sistema real de IBM**

Probamos ahora la ejecución en un sistema real.

In [ ]:
# Descomenta y ejecuta si es preciso
'''
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="TU_API_KEY",          # la clave de 44 caracteres
    instance="TU_INSTANCIA",     # opcional: nombre o CRN de la instancia
    set_as_default=True
)
'''

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.executor_sampler import Sampler

service = QiskitRuntimeService()

# Elegimos el dispositivo real menos ocupado con un mínimo de 5 cúbits
backend_real = service.least_busy(operational=True, simulator=False, min_num_qubits=5)

print(backend_real.name, backend_real.num_qubits)

In [ ]:
# Transpilamos al simulador con ruido: las puertas del circuito tienen que ser
# las que llevan error asociado en el modelo
pm_real = generate_preset_pass_manager(optimization_level=1,
                                  backend=backend_real)
# El circuito transpilado
bell_real = pm_real.run(bell)

# El Sampler para el backend real
sampler_real = Sampler(mode=backend_real)

# lanza el trabajo
job = sampler_real.run([bell_real], shots=shots)
print("ID del trabajo:", job.job_id())

# Espera a que termine y obtiene los valores
counts_real = job.result()[0].data.meas.get_counts()

print(counts_real)
display(plot_histogram([counts_ideal, counts_ruido, counts_fake, counts_real],
                       legend=['Ideal', 'Modelo propio', 'Fake Backend', 'Real Backend']))


<a name="qaoa"></a>
<a name="ejercicio8"></a>
# **Ejercicio 8: QAOA con ruido**

Repetimos el Ejemplo 2 del notebook de QAOA, minimizando el Hamiltoniano:

$$
H_c = 3Z_2Z_0 - Z_2Z_1 + 2Z_0
$$

pero sustituyendo las primitivas ideales (`StatevectorEstimator`, `StatevectorSampler`) por las de Qiskit Aer (`EstimatorV2`, `SamplerV2`) con un modelo de ruido. Ambas tienen la misma interfaz, por lo que las funciones `cost_func_estimator` y `circuit_sampler` no cambian.

 (a) Implementa dicha optimización usando un ruido diseñado con los errores que consideres. Compara el efecto del ruido con el ideal bajo distintas opciones de shots y distintos números de capas del circuito de QAOA. ¿Qué pasa al aumentar el número de capas del QAOA en simulación ideal? ¿Y en simulación ruidosa?

 (b) Elige un FakeBackend y repite el experimento anterior.

 (c) Ejecútalo en un equipo real de IBM

---



---



---

<a name="barren"></a>
# ***Barren plateaus***

Los algoritmos variacionales (VQE, QAOA, redes neuronales cuánticas...) minimizan una función de coste, que es el valor esperado de un Hamiltoniano $\langle H \rangle_{|\psi\rangle}$.

$$C(\boldsymbol{\theta}) = \langle\psi(\theta)|H|\psi(\theta)\rangle = \text{Tr}\left(\rho(\boldsymbol{\theta})H\right)$$

Usando un optimizador clásico, se estima el gradiente $\partial_k C = \partial C/\partial\theta_k$ mediante medidas.

Se dice que hay un ***barren plateau*** cuando el gradiente se concentra exponencialmente alrededor de 0 para el vector de parámetros $\boldsymbol{\theta}$:

$$
\mathbb{E}_{\boldsymbol{\theta}}\left[\partial_k C\right] = 0, \qquad \text{Var}_{\boldsymbol{\theta}}\left[\partial_k C\right] \in \mathcal{O}\left(b^{-n}\right),\quad b>1
$$

Al aumentar el número de cúbits $n$, el paisaje de coste se vuelve prácticamente plano y distinguir la dirección de descenso del ruido estadístico de las medidas requiere un número de *shots* $N$ que crece exponencialmente con $n$:

$$
\frac{1}{\sqrt{N}}\lesssim\text{std}_{\boldsymbol{\theta}}[\partial_k C] \in \mathcal{O} (b^{-n/2})
\quad\Longrightarrow\quad
N\in \Omega(b^n)
$$

siendo el $1/\sqrt{N}$ el orden del error de la estimación con $N$ *shots*.

# ***Barren plateaus***

Los algoritmos variacionales (VQE, QAOA, redes neuronales cuánticas...) minimizan una función de coste, que es el valor esperado de un Hamiltoniano $\langle H \rangle_{|\psi\rangle}$:

$$C(\boldsymbol{\theta}) = \langle\psi(\boldsymbol{\theta})|H|\psi(\boldsymbol{\theta})\rangle = \text{Tr}\left(\rho(\boldsymbol{\theta})H\right)$$

Usando un optimizador clásico, se estima el gradiente $\partial_k C = \partial C/\partial\theta_k$ mediante medidas.

Se dice que hay un ***barren plateau*** cuando el gradiente se concentra exponencialmente alrededor de 0 para el vector de parámetros $\boldsymbol{\theta}$. Siendo $n$ el número de cúbits y $b>1$ una constante, esto significa:

$$
\mathbb{E}_{\boldsymbol{\theta}}\left[\partial_k C\right] = 0, \qquad \text{Var}_{\boldsymbol{\theta}}\left[\partial_k C\right] \in \mathcal{O}\left(b^{-n}\right)
$$

Por la desigualdad de Chebyshev, $P\left(|\partial_k C|\ge\epsilon\right)\le\text{Var}[\partial_k C]/\epsilon^2$, por lo que casi ningún $\boldsymbol{\theta}$ tiene un gradiente apreciable: al aumentar $n$, el paisaje de coste se vuelve prácticamente plano.

Además, distinguir la dirección del descenso del ruido estadístico en las medidas requiere un número de *shots* $N$ que crece exponencialmente con $n$. Para poder resolver el gradiente, el error estadístico debe ser menor que su tamaño típico, que es su desviación típica $\sigma=\text{std}_{\boldsymbol{\theta}}[\partial_k C]=\sqrt{\text{Var}_{\boldsymbol{\theta}}[\partial_k C]}$:

$$
\frac{1}{\sqrt{N}}\lesssim\sigma .
$$

Como $\sigma\in\mathcal{O}\left(b^{-n/2}\right)$, resulta

$$
N\gtrsim\frac{1}{\sigma^2}\quad\Longrightarrow\quad N\in\Omega\left(b^{\,n}\right).
$$

Aquí $1/\sqrt{N}$ es el orden del error de la estimación con $N$ *shots* (para un observable con autovalores $\pm1$, como $Z_kZ_l$).

#### Causas

- **Expresividad** [arXiv:1803.11173](https://arxiv.org/abs/1803.11173): circuitos aleatorios profundos, tienen $\text{Var}[\partial_k C] \sim 2^{-n}$ (decrece exponencialmente con el número de cúbits).
- **Funciones de coste globales** [arXiv:2001.00550](https://arxiv.org/abs/2001.00550): observables que actúan sobre todos los cúbits (por ejemplo $Z^{\otimes n}$) producen *barren plateaus* incluso con circuitos poco profundos. Con observables locales (por ejemplo, $Z_0Z_1$) y circuitos de profundidad $\mathcal{O}(\log n)$ el gradiente decae solo polinómicamente
- **Exceso de entrelazamiento** [arXiv:2010.15968](https://arxiv.org/abs/2010.15968) entre los qubits visibles (sobre los que actúa el observable) y el resto del sistema (qubits ocultos): el estado reducido de los qubits visibles se acerca a $I/2^{n_V}$, siendo $n_V$ su número, y el coste deja de depender de $\boldsymbol{\theta}$.
- **Ruido** (*Noise-Induced Barren Plateaus*) [arXiv:2007.14384](https://arxiv.org/abs/2007.14384): con ruido de Pauli local antes y después de cada capa, que reduce cada componente de Pauli del estado por un factor $q<1$ (cuanto menor es $q$, más ruido), el gradiente decae exponencialmente con la profundidad $L$ del circuito:

$$|\partial_k C| \in \mathcal{O}\left(q^{L}\right)$$

El ruido lleva el estado hacia $I/2^n$ y el coste hacia $\text{Tr}(H)/2^n$, aplanando todo el paisaje.

#### Estrategias para mitigarlos

- Funciones de coste locales y ansatz poco profundos o inspirados en el problema (como QAOA).
- Inicializaciones inteligentes: *warm start*.
- Reducir el ruido: mejor hardware, transpilación que minimice el número de puertas de 2 cúbits y técnicas de mitigación de errores.

## Ejemplo: varianza del gradiente en función del ruido

Usamos un ansatz *hardware-efficient* de $n=4$ cúbits con $L$ capas de puertas $R_y$ y una cadena de $CX$, y un observable **local** $H = Z_0Z_1$ (para que, sin ruido, no haya un *barren plateau* inducido por el coste).

Para calcular el gradiente usamos la regla del **parameter-shift rule**, exacta para puertas de rotación $R_\sigma(\theta)=e^{-i\theta\sigma/2}$:

$$
\partial_k C(\boldsymbol{\theta}) = \frac{C\left(\boldsymbol{\theta} + \frac{\pi}{2}\boldsymbol{e}_k\right) - C\left(\boldsymbol{\theta} - \frac{\pi}{2}\boldsymbol{e}_k\right)}{2}
$$

donde $\boldsymbol e_k$ es el vector unitario en la dirección del parámetro $\theta_k$, de modo que solo se desplaza ese parámetro.

In [ ]:
from qiskit.circuit import ParameterVector

def ansatz_hea(n, capas):
  theta = ParameterVector('θ', n * capas)
  qc = QuantumCircuit(n)
  for l in range(capas):
    # Capa de rotaciones
    for i in range(n):
      qc.ry(theta[l * n + i], i)
    # Capa de entrelazamiento
    for i in range(n - 1):
      qc.cx(i, i + 1)
  return qc

ansatz_hea(4, 2).draw('mpl')

In [ ]:
# Función que estima la varianza del gradiente respecto a θ_k para parámetros aleatorios
def varianza_gradiente(circuito, observable, estimador, n_muestras=100, k=0, seed=0):

  rng = np.random.default_rng(seed)
  # Creamos n_muestras vectores de parámetros θ de componentes uniformes en $[0,2\pi)$.
  thetas = rng.uniform(0, 2 * np.pi, size=(n_muestras, circuito.num_parameters))

  # Desplazamiento ±π/2 en el parámetro k
  desplazamiento = np.zeros(circuito.num_parameters)
  desplazamiento[k] = np.pi / 2

  # Apilamos los parámetros: las primeras 100 filas son los desplazados hacia arriba y las 100 siguientes hacia abajo
  params = np.vstack([thetas + desplazamiento, thetas - desplazamiento])

  # Un único PUB con todos los conjuntos de parámetros: evs tiene 2*n_muestras valores
  evs = estimador.run([(circuito, observable, params)]).result()[0].data.evs

  # Obtiene los gradientes
  gradientes = (evs[:n_muestras] - evs[n_muestras:]) / 2

  # Devuelve la varianza
  return np.var(gradientes)

Creamos modelos de ruido despolarizante de intensidad creciente ($p$ en las puertas $CX$ y $p/10$ en las de 1 cúbit) y calculamos la varianza del gradiente para distintas profundidades: $\pi$

In [ ]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

def crear_modelo_despolarizante(p):
  modelo = NoiseModel()
  if p > 0:
    # No incluye la puerta rz ya que es una puerta virtual
    modelo.add_all_qubit_quantum_error(depolarizing_error(p / 10, 1), ['sx', 'x'])
    modelo.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx'])
  return modelo

n_bp = 4
observable_bp = SparsePauliOp("IIZZ")   # Z_0 Z_1
ruidos = [0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
capas_lista = [2, 4, 8]

pm_bp = generate_preset_pass_manager(optimization_level=1, basis_gates=['rz', 'sx', 'x', 'cx'])

varianzas = {}
for capas in capas_lista:
  circuito_bp = pm_bp.run(ansatz_hea(n_bp, capas))
  varianzas[capas] = []
  for p in ruidos:
    estimador_bp = AerEstimator(options={
        "backend_options": {"method": "density_matrix", "noise_model": crear_modelo_despolarizante(p)}
    })
    varianzas[capas].append(varianza_gradiente(circuito_bp, observable_bp, estimador_bp))
  print("L = {}: ".format(capas) + ", ".join("{:.2e}".format(v) for v in varianzas[capas]))

In [ ]:
# Representamos la varianza del gradiente frente al ruido
plt.figure(figsize=(12, 6))
for capas, vars_capas in varianzas.items():
  plt.semilogy(ruidos, vars_capas, 'o-', label="L = {} capas".format(capas))
plt.xlabel("Error despolarizante p (puertas CX)")
plt.ylabel(r"Var$[\partial_0 C]$")
plt.legend()
plt.show()

Observamos que:

- La varianza del gradiente **disminuye exponencialmente al aumentar el ruido**.
- El decaimiento es más rápido cuanto **más profundo** es el circuito.
- Sin ruido, la varianza también se reduce con $L$: el circuito se vuelve más expresivo y se acerca al régimen de *barren plateau* por expresividad [arXiv:1803.11173](https://arxiv.org/abs/1803.11173).

---



---



---

#### Referencias e información útil.

  - Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum Information*. Cambridge University Press. Capítulo 8: *Quantum noise and quantum operations*
  - Preskill, J. (2018). Quantum Computing in the NISQ era and beyond. Quantum, 2, 79. [arXiv:1801.00862](https://arxiv.org/abs/1801.00862)
  - Tutorial de Qiskit Aer, *Building Noise Models*: https://qiskit.github.io/qiskit-aer/tutorials/3_building_noise_models.html
  - Tutorial de Qiskit Aer, *Device backend noise model simulations*: https://qiskit.github.io/qiskit-aer/tutorials/2_device_noise_simulation.html
  - McClean, J. R., et al. (2018). Barren plateaus in quantum neural network training landscapes. Nature Communications, 9, 4812. [arXiv:1803.11173](https://arxiv.org/abs/1803.11173)
  - Cerezo, M., et al. (2021). Cost function dependent barren plateaus in shallow parametrized quantum circuits. Nature Communications, 12, 1791. [arXiv:2001.00550](https://arxiv.org/abs/2001.00550)
  - Wang, S., et al. (2021). Noise-induced barren plateaus in variational quantum algorithms. Nature Communications, 12, 6961. [arXiv:2007.14384](https://arxiv.org/abs/2007.14384)
  - Grant, E., et al. (2019). An initialization strategy for addressing barren plateaus in parametrized quantum circuits. Quantum, 3, 214. [arXiv:1903.05076](https://arxiv.org/abs/1903.05076)
  - Larocca, M., et al. (2025). Barren plateaus in variational quantum computing. Nature Reviews Physics, 7, 174-189. [arXiv:2405.00781](https://arxiv.org/abs/2405.00781)
  - Backends falsos de Qiskit IBM Runtime: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/fake-provider